STEP 1 — Fetch 1-minute bar data from Alpaca
=============================================
Saves sample QQQ and TQQQ data to CSV files for use in the backtest.

Setup:
    pip install alpaca-py pandas

Alpaca account (free):
    Sign up at https://alpaca.markets -> go to paper trading dashboard
    -> grab API Key ID and Secret Key and paste them below.

In [ ]:
import os
import pandas as pd
from datetime import datetime
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# ── CONFIG ────────────────────────────────────────────────────────────────────
API_KEY    = ""       # paste key here
API_SECRET = ""    # paste secret here

TICKERS    = ["QQQ", "TQQQ"]
START_DATE = datetime(2018, 1, 2)
END_DATE   = datetime(2025, 9, 28)
OUTPUT_DIR = "."        # saves CSVs in the same folder
# ─────────────────────────────────────────────────────────────────────────────


def fetch_and_save(ticker: str, client: StockHistoricalDataClient) -> None:
    output_path = os.path.join(OUTPUT_DIR, f"{ticker}_1min.csv")

    if os.path.exists(output_path):
        print(f"[{ticker}] CSV already exists, skipping download.")
        return

    print(f"[{ticker}] Downloading 1-min bars {START_DATE.date()} → {END_DATE.date()} ...")

    request = StockBarsRequest(
        symbol_or_symbols=ticker,
        timeframe=TimeFrame.Minute,
        start=START_DATE,
        end=END_DATE,
        adjustment="raw",
    )

    bars = client.get_stock_bars(request)
    df   = bars.df

    # Flatten multi-index -> just timestamp as index
    if isinstance(df.index, pd.MultiIndex):
        df = df.xs(ticker, level="symbol")

    # Ensure index is tz-aware Eastern time
    df.index = pd.to_datetime(df.index)
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    df.index = df.index.tz_convert("America/New_York")

    df.to_csv(output_path)
    print(f"[{ticker}] Saved {len(df):,} rows → {output_path}")


if __name__ == "__main__":
    if "YOUR_ALPACA" in API_KEY:
        raise ValueError(
            "Replace API_KEY and API_SECRET with your Alpaca credentials.\n"
            "Sign up at https://alpaca.markets"
        )

    client = StockHistoricalDataClient(API_KEY, API_SECRET)

    for ticker in TICKERS:
        fetch_and_save(ticker, client)

    print("\nAll done! :) run 2_backtest.py next.")


VWAP Trend Trading Backtest — no vectorbt
=======================================================
Based on: "Volume Weighted Average Price (VWAP) – The Holy Grail for Day
           Trading Systems" — Zarattini & Aziz (2023)

Python 3.8–3.12+. No exotic dependencies.

Setup:
    pip install alpaca-py pandas plotly

Usage:
    python 2_backtest.py    # QQQ (default)
    python 2_backtest.py --ticker TQQQ

In [ ]:
import argparse
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── CONFIG ────────────────────────────────────────────────────────────────────
TICKER          = "QQQ"
INITIAL_CAPITAL = 25_000.0
COMMISSION      = 0.0005      # $0.0005 per share (IB rate from paper)
DATA_DIR        = "."

ENTRY_HOUR      = 9
ENTRY_MIN       = 31          # 1st actionable bar after open
CLOSE_HOUR      = 15
CLOSE_MIN       = 59          # force-close 1 min before 4 PM
# ─────────────────────────────────────────────────────────────────────────────


# ══════════════════════════════════════════════════════════════════════════════
#  1. LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_data(ticker: str) -> pd.DataFrame:
    path = f"{DATA_DIR}/{ticker}_1min.csv"
    print(f"Loading {path} ...")
    df = pd.read_csv(path, index_col=0, parse_dates=True)

    # Ensure the index is a DatetimeIndex with timezone info
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, utc=True)
    
    if df.index.tz is None:
        df.index = df.index.tz_localize("America/New_York")
    else:
        df.index = df.index.tz_convert("America/New_York")

    df.columns = df.columns.str.lower()
    df = df[["open", "high", "low", "close", "volume"]].copy()
    df = df.sort_index()

    # Keep only regular trading hours 09:30–16:00
    df = df.between_time("09:30", "16:00")

    print(f"Loaded {len(df):,} bars  |  {df.index[0].date()} → {df.index[-1].date()}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
#  2. SESSION VWAP  (resets every trading day)
# ══════════════════════════════════════════════════════════════════════════════
def compute_vwap(df: pd.DataFrame) -> pd.Series:
    typical  = (df["high"] + df["low"] + df["close"]) / 3.0
    tpv      = typical * df["volume"]
    date_grp = df.index.normalize()
    vwap     = tpv.groupby(date_grp).cumsum() / df["volume"].groupby(date_grp).cumsum()
    return vwap


# ══════════════════════════════════════════════════════════════════════════════
#  3. BACKTEST LOOP (iterates day-by-day)
# ══════════════════════════════════════════════════════════════════════════════
def run_backtest(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    vwap = compute_vwap(df)
    df   = df.copy()
    df["vwap"] = vwap

    cash        = INITIAL_CAPITAL
    shares      = 0          # positive; long, negative; short
    position    = 0          # +1 long, -1 short, 0 flat
    nav_history = []         # (timestamp, nav) per bar
    trade_log   = []         # one row per completed trade

    entry_price = 0.0
    entry_time  = None

    trading_days = df.index.normalize().unique()
    total_days   = len(trading_days)

    for i, day in enumerate(trading_days):
        if i % 100 == 0:
            print(f"  Processing day {i+1}/{total_days}  ({day.date()})  NAV=${cash + shares * df.loc[df.index.normalize()==day, 'close'].iloc[-1] if shares != 0 else cash:,.0f}")

        day_df = df[df.index.normalize() == day].copy()

        first_bar = True

        for ts, row in day_df.iterrows():
            bar_hour = ts.hour
            bar_min  = ts.minute
            price    = row["close"]
            vwap_val = row["vwap"]

            # Current NAV
            mark_value = shares * price if shares != 0 else 0
            nav        = cash + mark_value
            nav_history.append((ts, nav))

            # Skip bars before entry time
            in_window    = (bar_hour > ENTRY_HOUR or (bar_hour == ENTRY_HOUR and bar_min >= ENTRY_MIN))
            is_eod       = (bar_hour == CLOSE_HOUR and bar_min == CLOSE_MIN)
            after_window = bar_hour >= 16

            if not in_window or after_window:
                continue

            if pd.isna(vwap_val) or vwap_val == 0:
                continue

            signal = 1 if price > vwap_val else (-1 if price < vwap_val else 0)
            if signal == 0:
                continue

            # ── FORCE CLOSE at end of day ──
            if is_eod and position != 0:
                proceeds = shares * price
                comm     = abs(shares) * COMMISSION
                if position == 1:
                    cash = cash - comm + proceeds
                else:
                    # short: buy back shares
                    cash -= abs(shares) * price + comm
                trade_log.append({
                    "entry_time" : entry_time,
                    "exit_time"  : ts,
                    "direction"  : "LONG" if position == 1 else "SHORT",
                    "entry_price": entry_price,
                    "exit_price" : price,
                    "shares"     : abs(shares),
                    "pnl"        : (price - entry_price) * abs(shares) * position - abs(shares) * COMMISSION * 2,
                })
                shares   = 0
                position = 0
                continue

            # ── ENTRY or REVERSAL ──
            if signal != position:
                # Close existing position first
                if position != 0:
                    comm = abs(shares) * COMMISSION
                    pnl  = (price - entry_price) * abs(shares) * position - abs(shares) * COMMISSION * 2
                    cash += shares * price - comm if position == 1 else -(abs(shares) * price + comm)
                    trade_log.append({
                        "entry_time" : entry_time,
                        "exit_time"  : ts,
                        "direction"  : "LONG" if position == 1 else "SHORT",
                        "entry_price": entry_price,
                        "exit_price" : price,
                        "shares"     : abs(shares),
                        "pnl"        : pnl,
                    })
                    shares   = 0
                    position = 0

                # Open new position using 100% of available cash
                new_shares  = int(cash / price)
                if new_shares == 0:
                    continue

                entry_comm  = new_shares * COMMISSION
                if signal == 1:       # long
                    cost     = new_shares * price + entry_comm
                    cash    -= cost
                    shares   = new_shares
                else:                 # short
                    cash    += new_shares * price - entry_comm  # receive proceeds, pay commission
                    shares   = -new_shares  # negative = short

                position    = signal
                entry_price = price
                entry_time  = ts
                first_bar   = False

        # EOD: if position still open and EOD bar wasn't hit, close anyway
        if position != 0 and len(day_df) > 0:
            last_price = day_df["close"].iloc[-1]
            last_ts    = day_df.index[-1]
            comm       = abs(shares) * COMMISSION
            pnl        = (last_price - entry_price) * abs(shares) * position - abs(shares) * COMMISSION * 2
            cash      += shares * last_price - comm if position == 1 else -(abs(shares) * last_price + comm)
            trade_log.append({
                "entry_time" : entry_time,
                "exit_time"  : last_ts,
                "direction"  : "LONG" if position == 1 else "SHORT",
                "entry_price": entry_price,
                "exit_price" : last_price,
                "shares"     : abs(shares),
                "pnl"        : pnl,
            })
            shares   = 0
            position = 0

    nav_df   = pd.DataFrame(nav_history, columns=["timestamp", "nav"]).set_index("timestamp")
    trade_df = pd.DataFrame(trade_log)
    return nav_df, trade_df, cash


# ══════════════════════════════════════════════════════════════════════════════
#  4. BUY & HOLD BENCHMARK
# ══════════════════════════════════════════════════════════════════════════════
def buy_and_hold(df: pd.DataFrame) -> pd.Series:
    first_price  = df["close"].iloc[0]
    shares_held  = int(INITIAL_CAPITAL / first_price)
    leftover     = INITIAL_CAPITAL - shares_held * first_price
    nav          = df["close"] * shares_held + leftover
    return nav


# ══════════════════════════════════════════════════════════════════════════════
#  5. PERFORMANCE METRICS
# ══════════════════════════════════════════════════════════════════════════════
def calc_metrics(nav: pd.Series, label: str, n_trades: int = None):
    daily_nav   = nav.resample("1D").last().dropna()
    daily_ret   = daily_nav.pct_change().dropna()

    total_ret   = (nav.iloc[-1] / INITIAL_CAPITAL - 1) * 100
    n_years     = (nav.index[-1] - nav.index[0]).days / 365.25
    ann_ret     = ((nav.iloc[-1] / INITIAL_CAPITAL) ** (1 / n_years) - 1) * 100
    ann_vol     = daily_ret.std() * np.sqrt(252) * 100
    sharpe      = (ann_ret / ann_vol) if ann_vol > 0 else 0

    roll_max    = daily_nav.cummax()
    drawdown    = (daily_nav - roll_max) / roll_max
    mdd         = drawdown.min() * 100

    print(f"\n{'='*58}")
    print(f"  {label}")
    print(f"{'='*58}")
    print(f"  End Value           : ${nav.iloc[-1]:>12,.0f}")
    print(f"  Total Return        : {total_ret:>10.1f}%")
    print(f"  Avg Annual Return   : {ann_ret:>10.1f}%")
    print(f"  Annualised Vol      : {ann_vol:>10.1f}%")
    print(f"  Sharpe Ratio        : {sharpe:>10.2f}")
    print(f"  Max Drawdown        : {mdd:>10.1f}%")
    if n_trades is not None:
        print(f"  Total Trades        : {n_trades:>10,}")
    print(f"{'='*58}")

    return {"total_ret": total_ret, "ann_ret": ann_ret, "sharpe": sharpe, "mdd": mdd}


# ══════════════════════════════════════════════════════════════════════════════
#  6. PLOT
# ══════════════════════════════════════════════════════════════════════════════
def plot_results(nav_strategy: pd.Series, nav_bh: pd.Series, ticker: str):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=nav_strategy.index, y=nav_strategy["nav"],
        name=f"VWAP TT ({ticker})", line=dict(color="royalblue", width=2)
    ))
    fig.add_trace(go.Scatter(
        x=nav_bh.index, y=nav_bh.values,
        name=f"Buy & Hold ({ticker})", line=dict(color="black", width=1, dash="dot")
    ))
    fig.update_layout(
        title      = f"VWAP Trend Trading vs Buy & Hold — {ticker}",
        yaxis_title= "Portfolio Value ($)",
        xaxis_title= "Date",
        hovermode  = "x unified",
        template   = "plotly_white",
    )
    out = f"equity_curve_{ticker}.html"
    fig.write_html(out)
    print(f"\nEquity curve saved → {out}  (open in your browser)")


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--ticker", default=TICKER, help="QQQ or TQQQ")
    args   = parser.parse_args()
    ticker = args.ticker.upper()

    df = load_data(ticker)

    # ── Strategy ──
    print(f"\nRunning VWAP Trend Trading backtest on {ticker} ...")
    nav_df, trade_df, final_cash = run_backtest(df, ticker)

    # ── Benchmark ──
    bh_nav = buy_and_hold(df)

    # ── Metrics ──
    calc_metrics(nav_df["nav"], f"VWAP Trend Trading — {ticker}", n_trades=len(trade_df))
    calc_metrics(bh_nav,        f"Buy & Hold         — {ticker}")

    # ── Trade stats ──
    if len(trade_df) > 0:
        wins     = trade_df[trade_df["pnl"] > 0]
        losses   = trade_df[trade_df["pnl"] <= 0]
        hit_rate = len(wins) / len(trade_df) * 100
        print(f"\nTrade breakdown:")
        print(f"  Total trades  : {len(trade_df):,}")
        print(f"  Win rate      : {hit_rate:.1f}%")
        print(f"  Avg win  $PnL : ${wins['pnl'].mean():,.2f}" if len(wins) > 0 else "  No wins")
        print(f"  Avg loss $PnL : ${losses['pnl'].mean():,.2f}" if len(losses) > 0 else "  No losses")
        print(f"  Best trade    : ${trade_df['pnl'].max():,.2f}")
        print(f"  Worst trade   : ${trade_df['pnl'].min():,.2f}")
        gain_loss = abs(wins["pnl"].mean() / losses["pnl"].mean()) if len(losses) > 0 else float("inf")
        print(f"  Gain:Loss     : {gain_loss:.2f}x")

    # ── Plot ──
    plot_results(nav_df, bh_nav, ticker)
